# 🤖 Veille Technique — BERT pour la Classification de Texte

## Contexte Projet 8 OpenClassrooms

Ce notebook met en œuvre une **technique récente de traitement du langage naturel** : **BERT**
(Bidirectional Encoder Representations from Transformers).

## 📄 Article de référence

> **Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2018)**
> *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*
> ArXiv: https://arxiv.org/abs/1810.04805

## 💡 Concept clé : Qu'est-ce que BERT ?

BERT est un modèle de langage **pré-entraîné** basé sur l'architecture **Transformer**.

| Caractéristique | Ancien (TF-IDF, LSTM) | BERT |
|---|---|---|
| Lecture du contexte | Unilatérale (gauche→droite) | **Bidirectionnelle** |
| Représentation des mots | Statique (sac de mots) | **Contextuelle** |
| Pré-entraînement | Non | **Oui (Wikipedia + BookCorpus)** |
| Fine-tuning | Non applicable | **Oui, sur tâche spécifique** |

**Exemple concret :**
- "Je vais à la *banque*" (institution) vs "Je m'assieds sur la *banque*" (siège)
- BERT comprend que "banque" a un sens différent selon le contexte
- TF-IDF traite "banque" de la même façon dans les deux phrases

## 🎯 Objectif de ce notebook

Comparer deux approches pour la **classification de sentiment** (positif/négatif) :

| Approche | Technique | Type |
|---|---|---|
| **Baseline** | TF-IDF + Régression Logistique | Classique |
| **Récente** | DistilBERT (version légère de BERT) | Transformer 2019 |

**Dataset :** IMDb Movie Reviews (50 000 avis de films)

## 🔗 Lien avec le projet de scoring crédit

BERT pourrait être appliqué à Home Credit pour :
- Analyser les **commentaires libres** des conseillers
- Classifier des **documents clients** (justificatifs, courriers)
- Détecter des **patterns dans les textes** de demandes de crédit


## 1. Installation des dépendances


In [ ]:
# Décommenter pour installer
# !pip install transformers datasets torch scikit-learn matplotlib pandas numpy

print('Pour installer les dépendances :')
print('pip install transformers datasets torch scikit-learn')

## 2. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# NLP classique (baseline)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# Hugging Face (BERT)
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import torch

print(f'✅ Imports OK')
print(f'PyTorch  : {torch.__version__}')
print(f'GPU dispo: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device   : {DEVICE}')

## 3. Chargement du dataset IMDb


In [ ]:
print('⏳ Chargement du dataset IMDb (avis de films)...')
dataset = load_dataset('imdb')

print(f'\n📊 Structure :')
print(dataset)
print(f'\nExemple :')
print(f'  Texte : {dataset["train"][0]["text"][:150]}...')
print(f'  Label : {dataset["train"][0]["label"]}  (0=négatif, 1=positif)')

In [ ]:
# Sous-ensemble pour aller plus vite (CPU friendly)
N_TRAIN = 2000
N_TEST  = 500

train_df = pd.DataFrame(dataset['train']).sample(N_TRAIN, random_state=42).reset_index(drop=True)
test_df  = pd.DataFrame(dataset['test']).sample(N_TEST,  random_state=42).reset_index(drop=True)

print(f'Train : {len(train_df)} exemples')
print(f'Test  : {len(test_df)} exemples')
print(f'\nDistribution train :')
print(train_df['label'].value_counts().rename({0: 'Négatif', 1: 'Positif'}))

y_train = train_df['label'].values
y_test  = test_df['label'].values

## 4. Approche Baseline — TF-IDF + Régression Logistique

Technique classique utilisée dans les projets précédents OpenClassrooms.
- **TF-IDF** : représente chaque texte comme un vecteur de fréquences de mots
- **Régression Logistique** : classifie à partir de ce vecteur

**Limite** : pas de contexte — "pas bien" et "bien" ont des vecteurs très différents
mais BERT comprendrait la négation.


In [ ]:
print('=' * 55)
print('  BASELINE : TF-IDF + Régression Logistique')
print('=' * 55)

# Vectorisation TF-IDF
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),    # unigrammes + bigrammes
    stop_words='english'
)
X_train_tfidf = vectorizer.fit_transform(train_df['text'])
X_test_tfidf  = vectorizer.transform(test_df['text'])

print(f'Matrice TF-IDF : {X_train_tfidf.shape}')
print(f'Nombre de features : {X_train_tfidf.shape[1]:,}')

In [ ]:
# Entraînement
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)

# Évaluation
y_pred_lr  = lr.predict(X_test_tfidf)
y_proba_lr = lr.predict_proba(X_test_tfidf)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)

print(f'\n📊 Résultats TF-IDF + Régression Logistique :')
print(f'  Accuracy : {acc_lr:.4f}')
print(f'  AUC-ROC  : {auc_lr:.4f}')
print(f'\n{classification_report(y_test, y_pred_lr, target_names=["Négatif", "Positif"])}')

## 5. Approche Récente — DistilBERT (2019)

**DistilBERT** est une version compressée de BERT :
- 40% moins de paramètres
- 60% plus rapide
- Conserve 97% des performances de BERT

**Principe du fine-tuning :**
1. Partir du modèle pré-entraîné (déjà "instruit" sur Wikipedia)
2. Ajouter une couche de classification
3. Ré-entraîner quelques epochs sur nos données spécifiques


In [ ]:
print('=' * 55)
print('  BERT : DistilBERT fine-tuning')
print('=' * 55)

MODEL_NAME = 'distilbert-base-uncased'

print('\n⏳ Chargement du tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Exemple de tokenisation
exemple = 'This movie was absolutely fantastic!'
tokens  = tokenizer(exemple, return_tensors='pt')
print(f'\nExemple de tokenisation :')
print(f'  Texte  : "{exemple}"')
print(f'  Tokens : {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])}')
print('\n[CLS] = début de séquence, [SEP] = fin de séquence')

In [ ]:
# Tokenisation du dataset
def tokenize(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

print('⏳ Tokenisation train/test...')
train_enc = tokenize(list(train_df['text']))
test_enc  = tokenize(list(test_df['text']))
print(f'✅ Forme input_ids train : {train_enc["input_ids"].shape}')

In [ ]:
# Dataset PyTorch
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_ds = IMDbDataset(train_enc, list(y_train))
test_ds  = IMDbDataset(test_enc,  list(y_test))
print(f'Dataset train : {len(train_ds)} | Dataset test : {len(test_ds)}')

In [ ]:
# Modèle
print('⏳ Chargement DistilBERT...')
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(DEVICE)

n_params = sum(p.numel() for p in bert_model.parameters())
print(f'✅ Modèle chargé — {n_params:,} paramètres')

In [ ]:
# Métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds  = np.argmax(logits, axis=-1)
    probas = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        'accuracy': accuracy_score(labels, preds),
        'auc'     : roc_auc_score(labels, probas),
    }

# Configuration entraînement
training_args = TrainingArguments(
    output_dir               = './bert_output',
    num_train_epochs         = 2,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps             = 100,
    weight_decay             = 0.01,
    logging_steps            = 50,
    evaluation_strategy      = 'epoch',
    save_strategy            = 'no',
    report_to                = 'none',
)

trainer = Trainer(
    model           = bert_model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = test_ds,
    compute_metrics = compute_metrics,
)

print('✅ Trainer configuré')
print('⚠️  Temps estimé : ~5-10 min sur CPU | ~1 min sur GPU')

In [ ]:
# Fine-tuning
print('🚀 Lancement du fine-tuning DistilBERT...')
trainer.train()
print('✅ Fine-tuning terminé !')

In [ ]:
# Évaluation finale
eval_results = trainer.evaluate()
acc_bert = eval_results['eval_accuracy']
auc_bert = eval_results['eval_auc']

print(f'\n📊 Résultats DistilBERT :')
print(f'  Accuracy : {acc_bert:.4f}')
print(f'  AUC-ROC  : {auc_bert:.4f}')

## 6. Comparaison finale


In [ ]:
# Tableau comparatif
comparison = pd.DataFrame([
    {
        'Approche'  : 'TF-IDF + Régression Logistique',
        'Type'      : 'Classique (baseline)',
        'Accuracy'  : acc_lr,
        'AUC-ROC'   : auc_lr,
        'Temps'     : '< 10 sec',
        'Paramètres': '~10 000 features'
    },
    {
        'Approche'  : 'DistilBERT (fine-tuning)',
        'Type'      : 'Récente — Transformer 2019',
        'Accuracy'  : acc_bert,
        'AUC-ROC'   : auc_bert,
        'Temps'     : '5-10 min CPU',
        'Paramètres': '66 millions'
    },
])

print('=' * 65)
print('  COMPARAISON FINALE')
print('=' * 65)
display(comparison.set_index('Approche').style
        .format({'Accuracy': '{:.4f}', 'AUC-ROC': '{:.4f}'})
        .highlight_max(subset=['Accuracy', 'AUC-ROC'], color='lightgreen'))

gain_acc = (acc_bert - acc_lr) * 100
gain_auc = (auc_bert - auc_lr) * 100
print(f'\n🚀 Gain BERT vs baseline :')
print(f'   Accuracy  : +{gain_acc:.1f} points')
print(f'   AUC-ROC   : +{gain_auc:.1f} points')

In [ ]:
# Graphique comparatif
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models = ['TF-IDF\n+ Log. Reg.', 'DistilBERT\n(fine-tuning)']
colors = ['#3498db', '#e74c3c']

# Accuracy
ax = axes[0]
bars = ax.bar(models, [acc_lr, acc_bert],
              color=colors, alpha=0.85, edgecolor='black')
ax.set_title('Accuracy (↑ meilleur)', fontsize=13, fontweight='bold')
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [acc_lr, acc_bert]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)

# AUC
ax = axes[1]
bars = ax.bar(models, [auc_lr, auc_bert],
              color=colors, alpha=0.85, edgecolor='black')
ax.set_title('AUC-ROC (↑ meilleur)', fontsize=13, fontweight='bold')
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [auc_lr, auc_bert]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)

plt.suptitle(
    'Comparaison : TF-IDF + LR vs DistilBERT\nDataset IMDb (analyse de sentiment)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../reports/veille_bert_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé : reports/veille_bert_comparison.png')

## 7. Conclusion

### Résultats attendus

| Approche | Accuracy | AUC-ROC | Temps |
|---|---|---|---|
| TF-IDF + Logistic Regression | ~0.87 | ~0.95 | < 10 sec |
| **DistilBERT (fine-tuning)** | **~0.92** | **~0.98** | 5-10 min |

### Ce que BERT apporte de nouveau

1. **Attention bidirectionnelle** : lit le contexte gauche ET droit simultanément
2. **Pré-entraînement massif** : 66M paramètres entraînés sur Wikipedia + BookCorpus
3. **Fine-tuning** : adaptation rapide à une tâche spécifique
4. **Représentations contextuelles** : le même mot = représentation différente selon contexte

### Quand utiliser BERT vs TF-IDF ?

| Critère | TF-IDF + LR | BERT |
|---|---|---|
| Peu de données (< 1000) | ✅ Préférable | ❌ Risque de surapprentissage |
| Beaucoup de données | ✅ Correct | ✅ Meilleur |
| Ressources limitées | ✅ Idéal | ❌ Gourmand |
| Contexte important | ❌ Limité | ✅ Excellent |
| Production temps réel | ✅ Rapide | ⚠️ Plus lent |

### Références

1. Devlin et al. (2018) — BERT — https://arxiv.org/abs/1810.04805
2. Sanh et al. (2019) — DistilBERT — https://arxiv.org/abs/1910.01108
3. Hugging Face Transformers — https://huggingface.co/docs/transformers
